# Creating simulation replicates and baseline benchmarking

## Population size change:

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep 10X | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//'))

mkdir -p ../replicates/population_size_change
for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
    for name in "${names_l[@]}"; do
        if [ ! -f ../replicates/population_size_change/p${p//./}-${name}/emission.gtrees ]; then
          # echo ${name}
          cat  ../inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
          python ../../yule-trees/scripts/simulate_mixture_condition.py \
            -x ../inferred_genetrees/concat-default.gtrees \
            -y ../inferred_genetrees/concat-${name}.gtrees \
            -p $p -r 0.99999 -o ../replicates/population_size_change/p${p//./}-${name}/
        fi
    done
done

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep 10X | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

for method in pbp-hmm cbp-hmm bcb-hmm dto-hmm mlt-hmm; do
    for name in "${names_l[@]}"; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            edge=$(echo ${name} | cut -f2 -d '-' | cut -f1 -d'_')
            output_path=../replicates/population_size_change/p${p//./}-${name}/pred-${method//-/}.txt
            if [ ! -f "$output_path" ]; then
                # echo ${output_path}
                python ../../../phlag/sbag.py $method \
                     -s ../main_neoaves-num_generations.tre \
                     -g ../replicates/population_size_change/p${p//./}-${name}/emission.gtrees \
                     -c ${edge} -o ${output_path} &
            fi
        done
    done
    wait
done

In [ ]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep 10X | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

rm -f ../results/baseline_metrics-population_size_change.tsv

for name in "${names_l[@]}"; do
    for method in pbp-hmm bcb-hmm dto-hmm mlt-hmm cbp-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            metrics=$(python ../../yule-trees/scripts/compute_metrics.py \
                    -x ../replicates/population_size_change/p${p//./}-${name}/pred-${method//-/}.txt \
                    -y ../replicates/population_size_change/p${p//./}-${name}/info.txt \
                    --method phlag --revert)
            if [ -n "${metrics}" ]; then
                edge=$(echo ${name} | cut -f2 -d '-' | cut -f1 -d'_')
                result=$(printf "${metrics//\\n/}\t${method}\t${name}\t${p}\t${edge}")
                echo ${result} >> ../results/baseline_metrics-population_size_change.tsv
            fi
        done
    done
done

## Recombination suppression

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_suppression | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

mkdir -p ../replicates/recombination_suppression

for name in "${names_l[@]}"; do
    cat  ../inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
    for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
        # if [ ! -f ../replicates/recombination_suppression/p${p//./}-${name}/emission.gtrees ]; then
            echo ${name} ${p}
            edge=$(echo ${name} | cut -f3 -d'-')
            if grep -w -q "${edge}" ../main_neoaves-num_generations.tre; then
                default_gtrees="../inferred_genetrees/concat-default.gtrees"
            else
                default_gtrees="../inferred_genetrees/concat-favian.gtrees"
            fi
            python ../../yule-trees/scripts/simulate_mixture_condition.py \
                -x ${default_gtrees} \
                -y ../inferred_genetrees/concat-${name}.gtrees \
                -p $p -r 0.99999 -o ../replicates/recombination_suppression/p${p//./}-${name}/
        # fi
    done
done

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_suppression | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag
rm -f commands-recombination_suppression.txt
for name in "${names_l[@]}"; do
    for method in pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            edge=$(echo ${name} | cut -f3 -d'-')
            output_path="../replicates/recombination_suppression/p${p//./}-${name}/pred-${method//-/}.txt"
            # if [ ! -f "${output_path}" ]; then
                # echo ${output_path}
                if grep -w -q "${edge}" ../main_neoaves-num_generations.tre; then
                    species_tree="../main_neoaves-num_generations.tre"
                else
                    species_tree="../main-num_generations.tre"
                fi
                echo "python ../../../phlag/sbag.py $method \
                     -s ${species_tree}\
                     -g ../replicates/recombination_suppression/p${p//./}-${name}/emission.gtrees \
                     -c ${edge} -o ${output_path}" >> commands-recombination_suppression.txt
            # fi
        done
    done
    # echo "Waiting for ${name}"
    # wait
done

In [ ]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_suppression | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

rm -f ../results/baseline_metrics-recombination_suppression.tsv

for name in "${names_l[@]}"; do
    for method in pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            metrics=$(python ../../yule-trees/scripts/compute_metrics.py \
                    -x ../replicates/recombination_suppression/p${p//./}-${name}/pred-${method//-/}.txt \
                    -y ../replicates/recombination_suppression/p${p//./}-${name}/info.txt \
                    --method phlag --revert)
            if [ -n "${metrics}" ]; then
                edge=$(echo ${name} | cut -f3 -d'-')
                result=$(printf "${metrics//\\n/}\t${method}\t${name}\t${p}\t${edge}")
                echo ${result} >> ../results/baseline_metrics-recombination_suppression.tsv
            fi
        done
    done
done

## Recombination increase

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_increase | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

mkdir -p ../replicates/recombination_increase

for name in "${names_l[@]}"; do
    cat  ../inferred_genetrees/${name}-500Kb* > ../inferred_genetrees/concat-${name}.gtrees
    for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
        # if [ ! -f ../replicates/recombination_increase/p${p//./}-${name}/emission.gtrees ]; then
            echo ${name} ${p}
            edge=$(echo ${name} | cut -f3 -d'-')
            if grep -w -q "${edge}" ../main_neoaves-num_generations.tre; then
                default_gtrees="../inferred_genetrees/concat-default.gtrees"
            else
                default_gtrees="../inferred_genetrees/concat-favian.gtrees"
            fi
            python ../../yule-trees/scripts/simulate_mixture_condition.py \
                -x ${default_gtrees} \
                -y ../inferred_genetrees/concat-${name}.gtrees \
                -p $p -r 0.99999 -o ../replicates/recombination_increase/p${p//./}-${name}/
        # fi
    done
done

In [ ]:
%%bash
declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_increase | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag
rm -f commands-recombination_increase.txt
for name in "${names_l[@]}"; do
    for method in pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            edge=$(echo ${name} | cut -f3 -d'-')
            output_path="../replicates/recombination_increase/p${p//./}-${name}/pred-${method//-/}.txt"
            # if [ ! -f "${output_path}" ]; then
                # echo ${output_path}
                if grep -w -q "${edge}" ../main_neoaves-num_generations.tre; then
                    species_tree="../main_neoaves-num_generations.tre"
                else
                    species_tree="../main-num_generations.tre"
                fi
                echo "python ../../../phlag/sbag.py $method \
                     -s ${species_tree}\
                     -g ../replicates/recombination_increase/p${p//./}-${name}/emission.gtrees \
                     -c ${edge} -o ${output_path}" >> commands-recombination_increase.txt
            # fi
        done
    done
    # echo "Waiting for ${name}"
    # wait
done

In [ ]:
%%bash
eval "$(micromamba shell hook --shell bash)"
micromamba activate phlag

declare -a names_l=($(ls ../inferred_genetrees/ | grep 500Kb | grep recombination_increase | cut -f1 -d'.' | sed 's/-500Kb-[0-9]//' | sort | uniq))

rm -f ../results/baseline_metrics-recombination_increase.tsv

for name in "${names_l[@]}"; do
    for method in pbp-hmm bcb-hmm dto-hmm mlt-hmm; do
        for p in 0.02 0.05 0.10 0.15 0.20 0.25; do
            echo $name $method $p
            metrics=$(python ../../yule-trees/scripts/compute_metrics.py \
                    -x ../replicates/recombination_increase/p${p//./}-${name}/pred-${method//-/}.txt \
                    -y ../replicates/recombination_increase/p${p//./}-${name}/info.txt \
                    --method phlag --revert)
            if [ -n "${metrics}" ]; then
                edge=$(echo ${name} | cut -f3 -d'-')
                result=$(printf "${metrics//\\n/}\t${method}\t${name}\t${p}\t${edge}")
                echo ${result} >> ../results/baseline_metrics-recombination_increase.tsv
            fi
        done
    done
done